In [1]:
!pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable


In [5]:
# Define the path to the dataset
data_dir = 'C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz'

In [6]:
import os
import numpy as np
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, InceptionV3, EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import joblib
import tensorflow as tf

In [25]:
# Define Constants
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 5
BEST_MODEL_PATH = "best_fish_model.keras"
DATA_DIR = "C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz/data/train"

In [26]:
# Data Generator

TRAIN_DIR = r"C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz/data/train"
VAL_DIR = r"C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz/data/val"
TEST_DIR = r"C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz/data/test"

train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 6225 images belonging to 11 classes.
Found 1092 images belonging to 11 classes.
Found 3187 images belonging to 11 classes.


In [27]:
# ⚖️ Class Weight Calculation
class_weight_vals = compute_class_weight(class_weight='balanced',
                                         classes=np.unique(train_gen.classes),
                                         y=train_gen.classes)
class_weight_dict = dict(enumerate(class_weight_vals))

In [28]:
print("class indices:", train_gen.class_indices)
print("num of classes;", len(train_gen.class_indices))

class indices: {'animal fish': 0, 'animal fish bass': 1, 'fish sea_food black_sea_sprat': 2, 'fish sea_food gilt_head_bream': 3, 'fish sea_food hourse_mackerel': 4, 'fish sea_food red_mullet': 5, 'fish sea_food red_sea_bream': 6, 'fish sea_food sea_bass': 7, 'fish sea_food shrimp': 8, 'fish sea_food striped_red_mullet': 9, 'fish sea_food trout': 10}
num of classes; 11


In [29]:
# 🧱 Base CNN (fallback option)
def create_cnn_model():
    model = Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(len(train_gen.class_indices), activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [30]:
# 🔁 Pretrained Model Builder
def create_pretrained_model(base_model_name):
    base_models = {
        "VGG16": VGG16,
        "ResNet50": ResNet50,
        "MobileNet": MobileNet,
        "InceptionV3": InceptionV3,
        "EfficientNetB0": EfficientNetB0
    }
    base_model = base_models[base_model_name](weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    base_model.trainable = True
    for layer in base_model.layers[:100]:  # Freeze first 100 layers for faster, stable training
        layer.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(len(train_gen.class_indices), activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [31]:
# 📚 Models Dictionary
models = {"CNN": create_cnn_model()}
for name in ["VGG16", "ResNet50", "MobileNet", "InceptionV3", "EfficientNetB0"]:
    models[name] = create_pretrained_model(name)

C:\Users\Chandra Shekar\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# 🚀 Training & Evaluation Loop
best_model = None
best_accuracy = 0
metrics_dict = {}

for name, model in models.items():
    print(f"\n🔧 Training {name}...\n")

    checkpoint = ModelCheckpoint(f"{name}_model.keras", save_best_only=True, monitor='val_accuracy', mode='max')
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(train_gen,
              validation_data=val_gen,
              epochs=EPOCHS,
              class_weight=class_weight_dict,
              callbacks=[checkpoint, early_stop],
              verbose=2)

    val_preds = np.argmax(model.predict(val_gen), axis=1)
    val_labels = val_gen.classes

    acc = accuracy_score(val_labels, val_preds)
    prec = precision_score(val_labels, val_preds, average='macro', zero_division=1)
    rec = recall_score(val_labels, val_preds, average='macro')
    f1 = f1_score(val_labels, val_preds, average='macro')
    cm = confusion_matrix(val_labels, val_preds)

    metrics_dict[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "Confusion Matrix": cm
    }

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name


🔧 Training CNN...

Epoch 1/5


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, InceptionV3, EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.callbacks import EarlyStopping

import pandas as pd
from sklearn.utils import class_weight

# Define the path to the dataset
data_dir = "C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz"

# Define parameters for image data generator
image_size = (224, 224) # Common size for VGG16 and other models
batch_size = 32

# Create data generators
# Assuming the data directory contains subdirectories for each class (e.g., data/class1, data/class2)
# You might need separate directories for training and validation data, e.g., data/train and data/validation
# For simplicity, let's assume a single directory with class subdirectories for now and split later or use validation_split
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # Using 20% of data for validation
)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training' # Specify training data
)

validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation' # Specify validation data
)

print("Data generators created.")
print("Number of classes:", train_generator.num_classes)
print("Class indices:", train_generator.class_indices)

In [ ]:
import os

data_dir = "C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz"

# Check if the data_dir path exists
if os.path.exists(data_dir):
    print(f"The data directory exists at: {data_dir}")

    # List the contents of the data_dir
    contents = os.listdir(data_dir)
    print("Contents of the data directory:")
    print(contents)

    # Optionally, check the number of items in a few subdirectories
    if contents:
        print("\nChecking item counts in subdirectories:")
        for item in contents[:5]: # Check the first 5 items (assuming they are subdirectories)
            item_path = os.path.join(data_dir, item)
            if os.path.isdir(item_path):
                num_items = len(os.listdir(item_path))
                print(f"- {item}: {num_items} items")
else:
    print(f"The data directory does NOT exist at: {data_dir}")


**Reasoning**:
Modify the data loading and model training steps to use the correct data path and adjust the `ImageDataGenerator` setup based on the identified train and validation subdirectories.



In [ ]:
data_dir = "C:/Users/Chandra Shekar/Downloads/Dataset/images.cv_jzk6llhf18tm3k0kyttxz"
train_dir = os.path.join(data_dir, 'train')
validation_dir = os.path.join(data_dir, 'val')

# Create data generators
# No need for validation_split as data is already in train and val folders
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    # Remove validation_split
)

validation_datagen = ImageDataGenerator(rescale=1./255) # Only rescale for validation data

train_generator = train_datagen.flow_from_directory(
    train_dir, # Point to the training directory
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical'
    # Remove subset='training'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir, # Point to the validation directory
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical'
    # Remove subset='validation'
)

print("Training data generator created.")
print("Validation data generator created.")
print("Number of classes found in training data:", train_generator.num_classes)
print("Number of classes found in validation data:", validation_generator.num_classes)